# Fair Job Recommender System (FJRS)
### Exposure Inequality Study — Pipeline Notebook

---

## Project Overview

This notebook is part of a study into **algorithmic fairness in job recommendation**. We examine whether a standard collaborative filtering model treats different groups of job seekers equitably — specifically whether certain groups receive systematically less exposure to high-opportunity jobs in ranked recommendation lists.

The project is split across two notebooks:

| Notebook | Purpose |
|---|---|
| `01_data_preparation.ipynb` | Clean the raw Kaggle dataset and export `fjrs_data.zip` |
| **`02_FJRS_pipeline.ipynb`** ← you are here | Load cleaned data, train model, measure and mitigate unfairness |

---

## What this notebook covers

| Step | Description | Status |
|---|---|---|
| **Step 0** | Setup & imports | ✅ Done |
| **Step 1** | Load cleaned data from `fjrs_data.zip` | ✅ Done |
| **Step 2** | Train / test split + interaction matrix | ✅ Done |
| Step 3 | Baseline model: Implicit Matrix Factorization (BPR-SGD) | ✅ Done |
| Step 4 | Generate baseline recommendations | ✅ Done |
| Step 5 | Accuracy metrics (Precision@K, Recall@K, nDCG@K) | ✅ Done |
| Step 6 | Exposure fairness metrics (position-based exposure parity) | ✅ Done |
| Step 7 | Fairness-aware re-ranking (greedy, tunable α/β) | ✅ Done |
| Step 8 | Evaluate fair model | ✅ Done |
| Step 9 | Compare baseline vs. fair model | ✅ Done |
| Step 10 | Trade-off frontier: sweep β | ✅ Done |
| Step 11 | Conclusion & next steps | ✅ Done |

---

## Key concepts

**Implicit feedback** — Users did not rate jobs. An application is treated as a binary positive signal (rating = 1). Non-applications are treated as unknown, not negative.

**Groups** — Derived from the `ManagedOthers` field in the user profile:
- **Group A** — users who have managed others (proxy for more experienced / privileged applicants)
- **Group B** — users who have not managed others

**Job tier** — Derived from job title keywords:
- **high_opportunity** — senior, lead, manager, engineer, analyst, etc.
- **standard** — all other roles

**Exposure** — Higher-ranked recommendations receive more attention. We model this with a position discount: `exposure(rank) = 1 / log₂(rank + 2)`. Fairness is measured as the gap in average exposure to high-opportunity jobs between Group A and Group B.

---

## Notebook handoff

Steps 0–2 are complete and documented below. Steps 3–11 contain working code with placeholder markdown. **Your task as the next contributor is to:**

1. Run the full notebook top-to-bottom to verify it executes correctly
2. Replace each `TODO` placeholder in the markdown cells with your own observations, interpretations and analysis
3. Add any additional experiments or visualisations you think are valuable
4. Update the Conclusion (Step 11) with your findings


---
## Step 0 — Setup & Imports

### What happens here
All third-party libraries are imported and global settings (random seed, plot style) are configured. A fixed `SEED = 42` is used throughout so every run produces identical results.

### Dependencies
```
numpy       — array operations, matrix math
pandas      — DataFrames for users, jobs, interactions
matplotlib  — plotting
seaborn     — plot styling
scipy       — sparse matrix (CSR format for the interaction matrix)
sklearn     — train/test split utility
zipfile     — reading the fjrs_data.zip bundle in-memory
```


In [ ]:
import json, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from io import BytesIO
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

SEED = 42
rng  = np.random.default_rng(SEED)
print('Setup complete')

---
## Step 1 — Load Cleaned Data

### What happens here
The cleaned dataset is loaded from `fjrs_data.zip`, which was produced by `01_data_preparation.ipynb`. All five artefacts are read directly into memory from the zip — no intermediate files are written to disk.

### Input
`fjrs_data.zip` containing:

| File | Description |
|---|---|
| `interactions.parquet` | One row per (user, job) application — columns: `user_id`, `job_id`, `rating` |
| `users.parquet` | One row per user — columns: `user_id`, `group` (`A` or `B`) |
| `jobs.parquet` | One row per job — columns: `job_id`, `tier` (`high_opportunity` or `standard`) |
| `user_history_agg.parquet` | Aggregated past job titles per user — not used in this pipeline but available for future content-based extensions |
| `meta.json` | Scalars `N_USERS` and `N_JOBS` (needed to initialise the MF model) |

### How to provide the zip
- **Colab** → a file picker opens automatically when you run the cell below
- **Jupyter / JupyterLab / Kaggle** → place `fjrs_data.zip` in the same folder as this notebook and run the cell

### Output variables
| Variable | Type | Shape | Description |
|---|---|---|---|
| `interactions` | DataFrame | (N_interactions, 3) | All user-job applications |
| `users` | DataFrame | (N_USERS, 2) | User group membership |
| `jobs` | DataFrame | (N_JOBS, 2) | Job tier labels |
| `N_USERS` | int | — | Total number of unique users |
| `N_JOBS` | int | — | Total number of unique jobs |


In [ ]:
# ── Locate fjrs_data.zip ─────────────────────────────────────────────────────
ZIP_NAME = 'fjrs_data.zip'

# Try Colab upload widget first; fall back to local file
zip_bytes = None
try:
    from google.colab import files
    print('Upload fjrs_data.zip using the file picker below:')
    uploaded = files.upload()
    zip_bytes = BytesIO(list(uploaded.values())[0])
    print('File received via Colab upload.')
except ImportError:
    # Jupyter / local — look for the zip next to this notebook
    local_zip = Path(ZIP_NAME)
    if not local_zip.exists():
        raise FileNotFoundError(
            f'{ZIP_NAME} not found. '
            'Place it in the same folder as this notebook, or run 01_data_preparation.ipynb first.'
        )
    zip_bytes = BytesIO(local_zip.read_bytes())
    print(f'Loaded {ZIP_NAME} from disk.')

# ── Extract into memory ───────────────────────────────────────────────────────
with zipfile.ZipFile(zip_bytes) as zf:
    def read_parquet(name):
        return pd.read_parquet(BytesIO(zf.read(name)))

    interactions = read_parquet('interactions.parquet')
    users        = read_parquet('users.parquet')
    jobs         = read_parquet('jobs.parquet')
    meta         = json.loads(zf.read('meta.json'))

N_USERS = meta['N_USERS']
N_JOBS  = meta['N_JOBS']

print(f'\nLoaded from {ZIP_NAME}:')
print(f'  interactions : {len(interactions):,} rows')
print(f'  users        : {len(users):,} rows')
print(f'  jobs         : {len(jobs):,} rows')
print(f'  N_USERS      : {N_USERS:,}')
print(f'  N_JOBS       : {N_JOBS:,}')
print()
print('Group distribution:')
print(users['group'].value_counts().to_string())
print()
print('Tier distribution:')
print(jobs['tier'].value_counts().to_string())

---
## Step 2 — Train / Test Split & Interaction Matrix

### What happens here
The interactions are randomly split 80/20 into a training set and a held-out test set. The training interactions are then stored as a **sparse CSR matrix** `R_train` of shape `(N_USERS, N_JOBS)`, where a value of 1 at position `[u, j]` means user `u` applied to job `j` during training.

### Why a sparse matrix?
With `N_USERS` × `N_JOBS` potentially reaching tens of millions of entries, a dense matrix would be impractical. Because most users only applied to a handful of jobs, the matrix is extremely sparse. CSR format stores only the non-zero entries, making both memory use and matrix operations efficient.

### Split strategy
We use a simple random split (`sklearn.model_selection.train_test_split`). A more rigorous approach would use a **temporal split** — training on older applications and testing on more recent ones — to better simulate a real deployment scenario. This is noted as a potential improvement in the conclusion.

### Output variables
| Variable | Description |
|---|---|
| `train_df` | 80% of interactions — used to train the model |
| `test_df` | 20% of interactions — used only for evaluation, never seen during training |
| `R_train` | Sparse CSR matrix `(N_USERS × N_JOBS)` built from `train_df` |


In [ ]:
train_df, test_df = train_test_split(
    interactions, test_size=0.2, random_state=SEED
)
print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}')

# Build sparse interaction matrix (train only)
R_train = csr_matrix(
    (train_df['rating'].values,
     (train_df['user_id'].values, train_df['job_id'].values)),
    shape=(N_USERS, N_JOBS)
)
print(f'Interaction matrix shape: {R_train.shape}')

---
## Step 3 — Baseline Model: Implicit Matrix Factorization (BPR-SGD)

### What happens here
We train a latent-factor model on the implicit feedback matrix using **Bayesian Personalized Ranking (BPR)** optimised with stochastic gradient descent (SGD).

### How BPR works
For each observed positive interaction `(u, i)`, we sample a random unobserved item `j` (a job the user did *not* apply to). The model is trained to rank `i` above `j` for user `u`. The loss is:

```
L = -log σ(x_ui - x_uj) + λ · ||θ||²
```

where `x_ui = P[u] · Q[i] + bias_u + bias_i` is the predicted score.

### Hyperparameters
| Parameter | Value | Description |
|---|---|---|
| `K_FACTORS` | 32 | Dimensionality of user/item latent vectors |
| `EPOCHS` | 15 | Number of full passes over the training data |
| `LR` | 0.05 | SGD learning rate |
| `REG` | 0.01 | L2 regularisation coefficient |

> **TODO — your turn:** After running the training, look at the convergence curve. Is the loss still decreasing at epoch 15? If so, consider increasing `EPOCHS`. Try a few different values of `K_FACTORS` (e.g. 16, 64) and note how it affects both training time and downstream metrics.


In [ ]:
class ImplicitMF:
    """Implicit Matrix Factorization with SGD (BPR-like)."""

    def __init__(self, n_users, n_items, k=32, lr=0.05, reg=0.01, seed=42):
        self.k = k
        self.lr = lr
        self.reg = reg
        rng = np.random.default_rng(seed)
        self.P = rng.normal(0, 0.1, (n_users, k))   # user factors
        self.Q = rng.normal(0, 0.1, (n_items, k))    # item factors
        self.user_bias = np.zeros(n_users)
        self.item_bias = np.zeros(n_items)
        self.global_bias = 0.0

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

    def fit(self, interactions_df, n_items, epochs=15, verbose=True):
        """BPR training: for each positive (u,i), sample negative j."""
        pos_pairs = interactions_df[['user_id', 'job_id']].values
        user_items = interactions_df.groupby('user_id')['job_id'].apply(set).to_dict()
        all_items = set(range(n_items))
        losses = []

        for epoch in range(epochs):
            epoch_loss = 0.0
            idx = np.random.permutation(len(pos_pairs))
            for ii in idx:
                u, i = pos_pairs[ii]
                # Sample negative item
                neg_pool = all_items - user_items.get(u, set())
                if not neg_pool:
                    continue
                j = np.random.choice(list(neg_pool))

                # Scores
                x_ui = self.P[u] @ self.Q[i] + self.user_bias[u] + self.item_bias[i]
                x_uj = self.P[u] @ self.Q[j] + self.user_bias[u] + self.item_bias[j]
                x_uij = x_ui - x_uj
                sig = self._sigmoid(-x_uij)

                # Update
                self.P[u]  += self.lr * (sig * (self.Q[i] - self.Q[j]) - self.reg * self.P[u])
                self.Q[i]  += self.lr * (sig * self.P[u] - self.reg * self.Q[i])
                self.Q[j]  += self.lr * (-sig * self.P[u] - self.reg * self.Q[j])
                self.user_bias[u] += self.lr * (sig - self.reg * self.user_bias[u])
                self.item_bias[i] += self.lr * (sig - self.reg * self.item_bias[i])
                self.item_bias[j] += self.lr * (-sig - self.reg * self.item_bias[j])

                epoch_loss += -np.log(self._sigmoid(x_uij) + 1e-10)

            avg_loss = epoch_loss / len(pos_pairs)
            losses.append(avg_loss)
            if verbose:
                print(f'  Epoch {epoch+1:2d}/{epochs}  BPR loss: {avg_loss:.4f}')

        return losses

    def predict_scores(self, user_id):
        """Return score vector for all items."""
        return self.P[user_id] @ self.Q.T + self.user_bias[user_id] + self.item_bias

    def recommend(self, user_id, topk=10, exclude=None):
        """Return top-K item indices."""
        scores = self.predict_scores(user_id)
        if exclude is not None:
            scores[list(exclude)] = -np.inf
        return np.argsort(scores)[::-1][:topk]

In [ ]:
# Train
EPOCHS = 15
K_FACTORS = 32
LR = 0.05
REG = 0.01

model = ImplicitMF(N_USERS, N_JOBS, k=K_FACTORS, lr=LR, reg=REG, seed=SEED)
print('Training baseline MF model...')
losses = model.fit(train_df, N_JOBS, epochs=EPOCHS, verbose=True)

In [ ]:
# Training curve
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS+1), losses, marker='o', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title('Training Convergence')
plt.tight_layout()
plt.show()

---
## Step 4 — Generate Baseline Recommendations

### What happens here
For every user we generate a ranked list of the top-`K` jobs the model thinks they are most likely to apply to, excluding jobs they already applied to in training.

### Parameters
| Parameter | Value | Description |
|---|---|---|
| `TOPK` | 10 | Number of recommendations per user |

### Exclusion of seen items
Items the user interacted with during training are masked (score set to `-inf`) before taking the top-K. This prevents the model from trivially "recommending" things the user has already seen.

> **TODO — your turn:** Inspect the recommendations for a few individual users. Do the titles of the recommended jobs look sensible given the user's group? You can join `baseline_recs` back to the original job titles using the job ID mapping from the preparation notebook.


In [ ]:
TOPK = 10

# Items each user interacted with in training (to exclude)
train_user_items = train_df.groupby('user_id')['job_id'].apply(set).to_dict()

# Ground truth: test items per user
test_user_items = test_df.groupby('user_id')['job_id'].apply(set).to_dict()

# Generate recommendations
baseline_recs = {}  # user_id -> list of job_ids
for uid in range(N_USERS):
    exclude = train_user_items.get(uid, set())
    baseline_recs[uid] = model.recommend(uid, topk=TOPK, exclude=exclude)

print(f'Generated top-{TOPK} recommendations for {N_USERS} users')

---
## Step 5 — Accuracy Metrics

### What happens here
We evaluate how accurately the model predicts which jobs a user will apply to, using the held-out test set as ground truth.

### Metrics
| Metric | Formula | What it measures |
|---|---|---|
| **Precision@K** | `\|recommended ∩ relevant\| / K` | Of the K recommendations, what fraction did the user actually apply to? |
| **Recall@K** | `\|recommended ∩ relevant\| / \|relevant\|` | Of all jobs the user applied to in the test set, what fraction did we recommend? |
| **nDCG@K** | DCG / IDCG | Ranking quality — relevant items appearing higher get more credit |

### Interpretation
These metrics are computed per user and averaged across all users. Higher is better for all three. Note that with sparse implicit data, absolute values tend to be low — what matters most is the relative comparison between baseline and fair model.

> **TODO — your turn:** Fill in the observed values once you have run the cell:
> - Precision@10: ___
> - Recall@10: ___
> - nDCG@10: ___
>
> Are these numbers better or worse than you expected for a simple MF model? How might you improve them?


In [ ]:
def precision_at_k(recs, truth, k):
    """Proportion of recommended items in top-K that are relevant."""
    if not truth:
        return 0.0
    return len(set(recs[:k]) & truth) / k


def recall_at_k(recs, truth, k):
    """Proportion of relevant items found in top-K."""
    if not truth:
        return 0.0
    return len(set(recs[:k]) & truth) / len(truth)


def ndcg_at_k(recs, truth, k):
    """Normalized Discounted Cumulative Gain."""
    if not truth:
        return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(recs[:k]) if item in truth)
    ideal_hits = min(len(truth), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def compute_accuracy_metrics(recs_dict, test_items_dict, k):
    """Compute mean accuracy metrics over all users."""
    precisions, recalls, ndcgs = [], [], []
    for uid, rec_list in recs_dict.items():
        truth = test_items_dict.get(uid, set())
        precisions.append(precision_at_k(rec_list, truth, k))
        recalls.append(recall_at_k(rec_list, truth, k))
        ndcgs.append(ndcg_at_k(rec_list, truth, k))
    return {
        f'Precision@{k}': np.mean(precisions),
        f'Recall@{k}': np.mean(recalls),
        f'nDCG@{k}': np.mean(ndcgs),
    }

In [ ]:
baseline_acc = compute_accuracy_metrics(baseline_recs, test_user_items, TOPK)
print('=== Baseline Accuracy ===')
for metric, val in baseline_acc.items():
    print(f'  {metric}: {val:.4f}')

---
## Step 6 — Exposure Fairness Metrics

### What happens here
We measure whether the baseline recommendations treat Group A and Group B equitably in terms of **exposure to high-opportunity jobs**.

### Exposure model
Items ranked higher receive more attention from users. We model this with a logarithmic position discount:

```
exposure(rank) = 1 / log₂(rank + 2)
```

So rank 0 (top position) has weight ≈ 1.0, rank 1 ≈ 0.63, rank 9 ≈ 0.29, etc.

For each user we sum the exposure weights of all high-opportunity jobs in their top-K list. We then compare the mean of this score across Group A vs Group B.

### Fairness metric
**Exposure Parity Gap** = `|mean_exposure_A − mean_exposure_B|`

A gap of 0 means both groups receive identical average exposure to high-opportunity jobs. A positive gap means Group A is advantaged.

### Why this matters
Even if the model is equally accurate for both groups, Group B users may systematically see fewer senior/specialist roles at the top of their recommendation list — reinforcing existing labour market inequalities.

> **TODO — your turn:** Fill in the observed values and write a short interpretation:
> - Group A mean exposure: ___
> - Group B mean exposure: ___
> - Parity gap: ___
>
> Is the gap large or small relative to the maximum possible exposure per user (sum of all position weights for K=10 ≈ 3.77)? What does the histogram shape tell you?


In [ ]:
def exposure_weight(rank):
    """Position-based exposure: w(rank) = 1 / log2(rank + 2)."""
    return 1.0 / np.log2(rank + 2)


def compute_exposure_metrics(recs_dict, users_df, jobs_df, topk):
    """
    Compute per-user exposure to high-opportunity jobs,
    then aggregate by group.
    """
    job_tier = jobs_df.set_index('job_id')['tier'].to_dict()
    user_group = users_df.set_index('user_id')['group'].to_dict()

    user_exposure = {}  # user_id -> total weighted exposure to high-opp jobs
    for uid, rec_list in recs_dict.items():
        exp = 0.0
        for rank, jid in enumerate(rec_list[:topk]):
            if job_tier.get(jid) == 'high_opportunity':
                exp += exposure_weight(rank)
        user_exposure[uid] = exp

    # Group-level
    group_exp = {'A': [], 'B': []}
    for uid, exp in user_exposure.items():
        g = user_group[uid]
        group_exp[g].append(exp)

    mean_A = np.mean(group_exp['A'])
    mean_B = np.mean(group_exp['B'])
    parity_gap = abs(mean_A - mean_B)

    return {
        'mean_exposure_A': mean_A,
        'mean_exposure_B': mean_B,
        'exposure_parity_gap': parity_gap,
        'group_exposures': group_exp,
    }

In [ ]:
baseline_exp = compute_exposure_metrics(baseline_recs, users, jobs, TOPK)

print('=== Baseline Exposure ===')
print(f"  Group A mean exposure to high-opp jobs: {baseline_exp['mean_exposure_A']:.4f}")
print(f"  Group B mean exposure to high-opp jobs: {baseline_exp['mean_exposure_B']:.4f}")
print(f"  Exposure Parity Gap:                    {baseline_exp['exposure_parity_gap']:.4f}")

In [ ]:
# Visualize exposure distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(baseline_exp['group_exposures']['A'], bins=30, alpha=0.6, label='Group A', color='#4C72B0')
axes[0].hist(baseline_exp['group_exposures']['B'], bins=30, alpha=0.6, label='Group B', color='#DD8452')
axes[0].axvline(baseline_exp['mean_exposure_A'], color='#4C72B0', ls='--', lw=2)
axes[0].axvline(baseline_exp['mean_exposure_B'], color='#DD8452', ls='--', lw=2)
axes[0].set_xlabel('Exposure to High-Opportunity Jobs')
axes[0].set_ylabel('# Users')
axes[0].set_title('Baseline: Exposure Distribution by Group')
axes[0].legend()

# Bar chart of means
means = [baseline_exp['mean_exposure_A'], baseline_exp['mean_exposure_B']]
axes[1].bar(['Group A', 'Group B'], means, color=['#4C72B0', '#DD8452'], edgecolor='white')
axes[1].set_ylabel('Mean Exposure to High-Opp Jobs')
axes[1].set_title(f"Baseline: Parity Gap = {baseline_exp['exposure_parity_gap']:.4f}")

plt.tight_layout()
plt.show()

---
## Step 7 — Fairness-Aware Re-ranking

### What happens here
A greedy post-processing algorithm re-orders each user's candidate list to increase fairness without fully discarding relevance. This is a **post-processing** approach — the underlying model is unchanged, only the final ranking is adjusted.

### Algorithm
For each user, we:
1. Retrieve a larger candidate pool (top-`n_candidates` = 50 from the base model)
2. Greedily select items one by one for each of the `TOPK` positions
3. At each step, pick the item with the highest combined score:

```
score(item) = α × relevance(item) + β × fairness_boost(item)
```

where:
- `relevance` = normalised MF score ∈ [0, 1]
- `fairness_boost` = 1.0 for high-opportunity items shown to Group B users, 0.3 for Group A, 0.0 otherwise
- `α` controls how much relevance matters
- `β` controls how aggressively we boost fairness

### Parameters
| Parameter | Value | Effect |
|---|---|---|
| `alpha` | 1.0 | Fixed relevance weight |
| `beta` | 0.6 | Fairness boost weight — increase to prioritise fairness over accuracy |
| `n_candidates` | 50 | Candidate pool size — larger pools give the algorithm more flexibility |

> **TODO — your turn:** Experiment with `beta`. What happens at `beta = 0`? At `beta = 2.0`? Does the algorithm converge to a point where increasing beta further stops reducing the gap?


In [ ]:
def fairness_rerank(model, users_df, jobs_df, train_user_items, topk=10,
                    n_candidates=50, alpha=1.0, beta=0.6):
    """
    Greedy fairness-aware re-ranking.
    
    Parameters
    ----------
    alpha : float
        Weight for relevance score.
    beta : float
        Weight for fairness boost.
    """
    job_tier = jobs_df.set_index('job_id')['tier'].to_dict()
    user_group = users_df.set_index('user_id')['group'].to_dict()
    n_users = len(users_df)

    # Boost high-opp items for group B (the disadvantaged group)
    fair_recs = {}
    for uid in range(n_users):
        exclude = train_user_items.get(uid, set())
        # Get larger candidate set
        candidates = model.recommend(uid, topk=n_candidates, exclude=exclude)
        scores = model.predict_scores(uid)

        # Normalize relevance scores to [0, 1]
        cand_scores = scores[candidates]
        s_min, s_max = cand_scores.min(), cand_scores.max()
        if s_max > s_min:
            norm_scores = (cand_scores - s_min) / (s_max - s_min)
        else:
            norm_scores = np.ones_like(cand_scores)

        # Greedy selection
        selected = []
        remaining = list(range(len(candidates)))

        for pos in range(topk):
            best_idx = None
            best_combined = -np.inf
            for idx in remaining:
                jid = candidates[idx]
                rel = norm_scores[idx]

                # Fairness boost: high-opp items get a boost for group B
                boost = 0.0
                if job_tier.get(jid) == 'high_opportunity' and user_group[uid] == 'B':
                    boost = 1.0
                elif job_tier.get(jid) == 'high_opportunity' and user_group[uid] == 'A':
                    boost = 0.3  # smaller boost for already-advantaged group

                combined = alpha * rel + beta * boost
                if combined > best_combined:
                    best_combined = combined
                    best_idx = idx

            selected.append(candidates[best_idx])
            remaining.remove(best_idx)

        fair_recs[uid] = np.array(selected)

    return fair_recs

In [ ]:
print('Running fairness-aware re-ranking (alpha=1.0, beta=0.6)...')
fair_recs = fairness_rerank(
    model, users, jobs, train_user_items,
    topk=TOPK, n_candidates=50, alpha=1.0, beta=0.6
)
print('Done')

---
## Step 8 — Evaluate Fair Recommendations

### What happens here
We apply the same accuracy and exposure metrics from Steps 5 and 6 to the re-ranked recommendations, so we can quantify what was gained in fairness and what was lost in accuracy.

> **TODO — your turn:** Fill in the observed values:
> - Fair Precision@10: ___  (baseline was ___)
> - Fair nDCG@10: ___  (baseline was ___)
> - Fair parity gap: ___  (baseline was ___)
>
> Does the accuracy drop seem acceptable given the fairness gain? How would you justify this trade-off to a business stakeholder?


In [ ]:
fair_acc = compute_accuracy_metrics(fair_recs, test_user_items, TOPK)
fair_exp = compute_exposure_metrics(fair_recs, users, jobs, TOPK)

print('=== Fair Model: Accuracy ===')
for metric, val in fair_acc.items():
    print(f'  {metric}: {val:.4f}')

print()
print('=== Fair Model: Exposure ===')
print(f"  Group A mean exposure: {fair_exp['mean_exposure_A']:.4f}")
print(f"  Group B mean exposure: {fair_exp['mean_exposure_B']:.4f}")
print(f"  Exposure Parity Gap:   {fair_exp['exposure_parity_gap']:.4f}")

---
## Step 9 — Compare: Baseline vs. Fair

### What happens here
A side-by-side comparison table and three visualisations summarise the effect of the fairness intervention across accuracy, exposure by group, and parity gap.

### How to read the plots
- **Left** — accuracy bars: a small drop is expected and acceptable
- **Middle** — exposure by group: the gap between Group A and B bars should shrink
- **Right** — parity gap: a lower bar means a fairer system

> **TODO — your turn:** Describe what you see in the three plots. Is the fairness improvement visible in the exposure-by-group chart? Does the delta column in the summary table match your expectation from Step 8?


In [ ]:
# Summary table
comparison = pd.DataFrame({
    'Metric': [
        f'Precision@{TOPK}', f'Recall@{TOPK}', f'nDCG@{TOPK}',
        'Exposure (Group A)', 'Exposure (Group B)', 'Parity Gap'
    ],
    'Baseline': [
        baseline_acc[f'Precision@{TOPK}'],
        baseline_acc[f'Recall@{TOPK}'],
        baseline_acc[f'nDCG@{TOPK}'],
        baseline_exp['mean_exposure_A'],
        baseline_exp['mean_exposure_B'],
        baseline_exp['exposure_parity_gap'],
    ],
    'Fair Re-ranked': [
        fair_acc[f'Precision@{TOPK}'],
        fair_acc[f'Recall@{TOPK}'],
        fair_acc[f'nDCG@{TOPK}'],
        fair_exp['mean_exposure_A'],
        fair_exp['mean_exposure_B'],
        fair_exp['exposure_parity_gap'],
    ],
})
comparison['Delta (Fair - Baseline)'] = comparison['Fair Re-ranked'] - comparison['Baseline']
comparison = comparison.round(4)
comparison

In [ ]:
# Visualization: side-by-side comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Accuracy comparison
acc_metrics = [f'Precision@{TOPK}', f'Recall@{TOPK}', f'nDCG@{TOPK}']
x = np.arange(len(acc_metrics))
w = 0.35
axes[0].bar(x - w/2, [baseline_acc[m] for m in acc_metrics], w, label='Baseline', color='#4C72B0')
axes[0].bar(x + w/2, [fair_acc[m] for m in acc_metrics], w, label='Fair', color='#55A868')
axes[0].set_xticks(x)
axes[0].set_xticklabels(acc_metrics)
axes[0].set_title('Accuracy: Baseline vs Fair')
axes[0].legend()

# Exposure by group
x2 = np.arange(2)
axes[1].bar(x2 - w/2, [baseline_exp['mean_exposure_A'], baseline_exp['mean_exposure_B']], w, label='Baseline', color='#4C72B0')
axes[1].bar(x2 + w/2, [fair_exp['mean_exposure_A'], fair_exp['mean_exposure_B']], w, label='Fair', color='#55A868')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(['Group A', 'Group B'])
axes[1].set_title('Exposure to High-Opp Jobs by Group')
axes[1].legend()

# Parity gap
axes[2].bar(['Baseline', 'Fair'],
            [baseline_exp['exposure_parity_gap'], fair_exp['exposure_parity_gap']],
            color=['#C44E52', '#55A868'], edgecolor='white')
axes[2].set_title('Exposure Parity Gap (lower = fairer)')
axes[2].set_ylabel('|mean_A - mean_B|')

plt.tight_layout()
plt.show()

---
## Step 10 — Trade-off Frontier: Sweeping β

### What happens here
We run the fairness re-ranker for eight values of `β` from 0.0 (pure relevance) to 2.0 (strong fairness push), recording both nDCG@10 and the parity gap at each point. The result is a **Pareto frontier** — a curve showing all achievable accuracy/fairness combinations.

### Why this matters
There is no single "right" value of β. Different applications require different operating points:
- A job board serving high-stakes placements may accept a larger accuracy drop for strong fairness guarantees
- A lightweight matching tool may need to keep accuracy high while making only modest fairness improvements

The frontier lets stakeholders make an informed choice rather than accepting whatever a single fixed β produces.

> **TODO — your turn:** Look at the frontier plot and identify:
> 1. The "knee" of the curve — the point where fairness improves rapidly without much accuracy loss
> 2. The point of diminishing returns — where further β increases cost more accuracy than they gain in fairness
>
> Which β would you recommend, and why? Add your reasoning here.


In [ ]:
betas = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 2.0]
frontier = []

for b in betas:
    print(f'beta = {b:.1f} ...', end=' ')
    recs_b = fairness_rerank(model, users, jobs, train_user_items,
                             topk=TOPK, n_candidates=50, alpha=1.0, beta=b)
    acc_b = compute_accuracy_metrics(recs_b, test_user_items, TOPK)
    exp_b = compute_exposure_metrics(recs_b, users, jobs, TOPK)
    frontier.append({
        'beta': b,
        f'nDCG@{TOPK}': acc_b[f'nDCG@{TOPK}'],
        f'Precision@{TOPK}': acc_b[f'Precision@{TOPK}'],
        'parity_gap': exp_b['exposure_parity_gap'],
        'mean_exp_A': exp_b['mean_exposure_A'],
        'mean_exp_B': exp_b['mean_exposure_B'],
    })
    print(f"nDCG={acc_b[f'nDCG@{TOPK}']:.4f}  gap={exp_b['exposure_parity_gap']:.4f}")

frontier_df = pd.DataFrame(frontier)
frontier_df

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

color1 = '#4C72B0'
color2 = '#C44E52'

ax1.set_xlabel('beta (fairness weight)', fontsize=12)
ax1.set_ylabel(f'nDCG@{TOPK}', color=color1, fontsize=12)
ax1.plot(frontier_df['beta'], frontier_df[f'nDCG@{TOPK}'],
         marker='o', color=color1, linewidth=2, label=f'nDCG@{TOPK}')
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
ax2.set_ylabel('Exposure Parity Gap', color=color2, fontsize=12)
ax2.plot(frontier_df['beta'], frontier_df['parity_gap'],
         marker='s', color=color2, linewidth=2, linestyle='--', label='Parity Gap')
ax2.tick_params(axis='y', labelcolor=color2)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=11)

plt.title('Accuracy-Fairness Trade-off Frontier', fontsize=14)
plt.tight_layout()
plt.show()

---
## Step 11 — Conclusion

### Summary of findings

> **TODO — your turn:** Replace the bullet points below with your actual observed results after running the notebook.

- The **baseline model** (pure collaborative filtering) amplifies existing interaction biases, giving Group A significantly more exposure to high-opportunity jobs than Group B.
- The **fairness-aware re-ranking** step reduces the exposure parity gap substantially. *(Fill in: by how much? From ___ to ___?)*
- There is a clear **accuracy-fairness trade-off**: increasing β reduces the parity gap but also decreases nDCG@K. *(Fill in: at the recommended β, accuracy dropped by ___% while the gap reduced by ___%)*
- The trade-off frontier shows a "knee" at β ≈ ___, making this a reasonable default operating point.

---

### Limitations & potential improvements

| Area | Current approach | Potential improvement |
|---|---|---|
| **Group definition** | Binary split on `ManagedOthers` | Use demographic attributes or learned user embeddings |
| **Tier definition** | Keyword matching on job title | Salary data, seniority level, or learned job embeddings |
| **Train/test split** | Random 80/20 | Temporal split to simulate real deployment |
| **Fairness notion** | Group exposure parity | Individual fairness, proportional exposure, or counterfactual fairness |
| **Mitigation** | Post-processing re-ranking | In-processing regularisation (add fairness penalty to BPR loss) |
| **Model** | BPR-SGD MF | Neural CF, two-tower models, LLM-based rankers |
| **Evaluation** | Offline metrics only | Online A/B test or user study |

---

### Next steps for the project

1. **Improve the group and tier definitions** — the current proxies are coarse; richer feature engineering would make the fairness analysis more meaningful
2. **Try in-processing fairness** — add a fairness regularisation term directly to the BPR objective instead of post-processing
3. **Temporal evaluation** — re-run the experiment with a time-based split to get a more realistic estimate of real-world performance
4. **Extend to individual fairness** — ensure similar users receive similar recommendations, not just aggregate group parity
5. **Scale up** — the current pipeline subsamples the data; validate findings on the full Kaggle dataset
